# Numbers and memory

Notebook 03 blamed part of Python's slowness on how it stores numbers, and showed that the order in which C reads memory changes its speed enormously. Both come down to how numbers are represented and stored.

## Everything is bits

Computer memory stores only **bits**: values that are either 0 or 1. Bits are grouped in eights, called **bytes**. You can picture memory as a very long row of numbered boxes, one byte each; the number of a box is its **address**.

Whole numbers are stored in binary, where each position is worth twice the previous one (1, 2, 4, 8…) instead of ten times, as in decimal. For example, $13 = 8 + 4 + 1$, which is `1101` in binary:

In [ ]:
format(13, "08b")

'00001101'

With $b$ bits there are $2^b$ different combinations. One byte can therefore store 256 values, 0 to 255, which is why pixel brightness in notebook 02 went from 0 to 255.

In [ ]:
{f"{bits} bits": f"{2**bits:,} values" for bits in [8, 16, 32, 64]}

{'8 bits': '256 values',
 '16 bits': '65,536 values',
 '32 bits': '4,294,967,296 values',
 '64 bits': '18,446,744,073,709,551,616 values'}

## Decimal numbers: floating point

Decimal numbers use an idea you already know from science: **scientific notation**. Avogadro's number is written $6.022 \times 10^{23}$: some significant digits (6.022) and an exponent that says where the decimal point goes.

**Floating-point** numbers do the same in binary. A fixed number of bits is split into three parts:

- a **sign** (positive or negative),
- the **significant digits** (called the *mantissa*),
- an **exponent** (where the point "floats" to).

Because the number of bits is fixed, the number of significant digits is limited. Every value is rounded to the nearest number the format can represent.

The three formats that matter in this series are named after their size in bits. `np.finfo` reports their properties, including roughly how many decimal digits of precision each one keeps:

In [ ]:
import numpy as np

{np.dtype(dtype).name: {"bits": np.finfo(dtype).bits, "decimal digits": np.finfo(dtype).precision,
                        "largest value": float(np.finfo(dtype).max)}
 for dtype in (np.float64, np.float32, np.float16)}

{'float64': {'bits': 64,
  'decimal digits': 15,
  'largest value': 1.7976931348623157e+308},
 'float32': {'bits': 32,
  'decimal digits': 6,
  'largest value': 3.4028234663852886e+38},
 'float16': {'bits': 16, 'decimal digits': 3, 'largest value': 65504.0}}

The same value, one third, stored in each format. Printed with 20 decimals, the rounding becomes visible:

In [ ]:
{np.dtype(dtype).name: f"{float(dtype(1 / 3)):.20f}" for dtype in (np.float64, np.float32, np.float16)}

{'float64': '0.33333333333333331483',
 'float32': '0.33333334326744079590',
 'float16': '0.33325195312500000000'}

Even `float64` is not exact. Some numbers that look simple in decimal, such as 0.1, have infinitely many digits in binary, just as $1/3 = 0.333\ldots$ does in decimal. So they are always rounded, and small errors appear:

In [ ]:
0.1 + 0.2, 0.1 + 0.2 == 0.3

(0.30000000000000004, False)

This is why decimal results must never be compared with `==`. Instead, we check that they are equal **within a small tolerance**. NumPy's `isclose` (for single values) and `allclose` (for whole arrays) do that, and this series uses them to check every result.

In [ ]:
np.isclose(0.1 + 0.2, 0.3)

np.True_

Fewer bits means less precision, but also **less memory**, and less data to move around. That trade-off matters a lot in AI. A model with 7 billion learned numbers (a medium-sized language model) needs very different amounts of memory depending on the format:

In [ ]:
parameter_count = 7_000_000_000
{np.dtype(dtype).name: f"{parameter_count * np.dtype(dtype).itemsize / 1e9:.0f} GB"
 for dtype in (np.float64, np.float32, np.float16)}

{'float64': '56 GB', 'float32': '28 GB', 'float16': '14 GB'}

Notebook 05 shows that lower precision also makes GPUs much faster.

## How Python stores a number

In C, a `double` (C's name for `float64`) is exactly 8 bytes: the bits of the number and nothing else.

In Python, **everything is an object**, including numbers. A Python `float` is a small structure in memory that contains:

- a **reference count**: how many variables currently point to it, so that Python can free the memory automatically when nobody uses it any more;
- a **pointer** to its type (a pointer is simply a memory address that says where something else is stored). This is how the interpreter knows, at run time, that the object is a float: the type check from notebook 01;
- finally, the 8 bytes of the value itself.

In [ ]:
import sys

sys.getsizeof(1.5), np.dtype(np.float64).itemsize

(24, 8)

24 bytes to store 8 bytes of information. This packaging is called **boxing**: the number is placed in a box with a label on it. The box has a cost in time too: every arithmetic result, such as `a * b`, needs a *new* box, which must be created in memory and later freed.

## Lists store pointers, arrays store numbers

A Python list does not contain the numbers themselves. It contains **pointers** to float objects, which can be anywhere in memory. A NumPy array stores the raw 8-byte values next to each other, with no boxes:

```text
Python list                                    NumPy array (float64)
┌─────┬─────┬─────┬─────┐                      ┌─────┬─────┬─────┬─────┐
│ ptr │ ptr │ ptr │ ptr │                      │ 1.5 │ 2.5 │ 3.5 │ 4.5 │   8 bytes each, side by side
└──┬──┴──┬──┴──┬──┴──┬──┘                      └─────┴─────┴─────┴─────┘
   ▼     ▼     ▼     ▼
 float float float float    ← separate 24-byte objects, scattered in memory
```

To read `values[2]` from a list, the computer reads the pointer, jumps to the object, checks its type and then reads the value. From an array, it computes *start address + 2 × 8* and reads the value directly.

In [ ]:
element_count = 1_000_000
values_list = [float(i) for i in range(element_count)]
values_array = np.arange(element_count, dtype=np.float64)

list_bytes = sys.getsizeof(values_list) + sum(sys.getsizeof(value) for value in values_list)
f"list: {list_bytes / 1e6:.0f} MB, array: {values_array.nbytes / 1e6:.0f} MB"

'list: 32 MB, array: 8 MB'

One caution: the array only avoids boxes as long as the work happens *inside* NumPy. Reading a single element from Python creates a box for it:

In [ ]:
type(values_array[0])

numpy.float64

So a Python `for` loop over a NumPy array is no faster than a loop over a list. NumPy is fast when a single call, such as `values_array.sum()` or `a @ b`, processes the whole array in compiled code. Notebook 04 builds on this.

## The memory hierarchy

A modern CPU performs an operation in a fraction of a nanosecond, but fetching a value from main memory (**RAM**) takes around 100 nanoseconds: hundreds of times longer. To avoid waiting, processors keep copies of recently used data in **caches**: small, very fast memories built into the processor chip. There are usually three levels, from smallest and fastest (L1) to largest and slowest (L3). Faster still are the **registers**, a few dozen storage slots inside each **core** where the actual arithmetic happens. (A core is one of the independent processing units inside a CPU; most CPUs have several. Notebook 04 returns to them.)

Think of a student working on an essay:

| Level | Analogy | Typical size | Typical access time |
|---|---|---|---|
| Registers | What you hold in your hands | Up to a few KB per core | Under 1 ns |
| L1 cache | Papers on your desk | 32–64 KB per core | About 1 ns |
| L2 cache | The shelf behind you | 1–2 MB per core | About 3–5 ns |
| L3 cache | The bookcase in the room | Tens of MB, shared | About 10–20 ns |
| RAM | A library across town | Tens of GB | About 80–100 ns |

The exact figures vary between processors; the orders of magnitude are what matter.

## Cache lines

Data does not travel from RAM to the cache one number at a time. It travels in blocks of 64 bytes called **cache lines**: eight `float64` values. Reading one number brings its seven neighbours along for free. Processors also notice when a program reads memory in order, and start fetching the next lines before they are requested.

So the *order* in which a program reads memory matters a great deal:

- reading **consecutive** values: one slow trip to RAM serves eight numbers, and the next trips are started early;
- **jumping around**: almost every read is a slow trip, and nothing can be predicted.

An experiment makes this visible. We read the same 20 million values twice: once in order, and once in a shuffled order. The amount of work is identical; only the order of the memory accesses changes. The array (160 MB) is much larger than any cache.

In [ ]:
from compare_efficiency.benchmark import time_function

value_count = 20_000_000
large_array = np.arange(value_count, dtype=np.float64)
in_order_positions = np.arange(value_count)
shuffled_positions = np.random.default_rng(0).permutation(value_count)

`large_array[positions]` reads the values at the given positions, in that order:

In [ ]:
in_order_seconds = time_function(lambda: large_array[in_order_positions].sum(), repeats=3)
shuffled_seconds = time_function(lambda: large_array[shuffled_positions].sum(), repeats=3)
f"in order: {in_order_seconds * 1000:.0f} ms, shuffled: {shuffled_seconds * 1000:.0f} ms " \
    f"({shuffled_seconds / in_order_seconds:.1f}x slower)"

'in order: 18 ms, shuffled: 105 ms (5.8x slower)'

Same numbers, same sum, same number of reads, and yet the shuffled version is several times slower. **How** a program walks through memory can matter as much as how many operations it performs. Notebook 03 shows this effect again inside matrix multiplication.

## The bits of a float

The standard that defines floating-point formats is called **IEEE 754**. The `struct` module can reveal the 64 bits that store the value 0.1:

In [ ]:
import struct

bits = format(struct.unpack(">Q", struct.pack(">d", 0.1))[0], "064b")
{"sign": bits[0], "exponent": bits[1:12], "mantissa": bits[12:]}

{'sign': '0',
 'exponent': '01111111011',
 'mantissa': '1001100110011001100110011001100110011001100110011010'}

The stored value is

$$(-1)^{\text{sign}} \times 1.\text{mantissa}_2 \times 2^{\,\text{exponent} - 1023}$$

- The sign bit is 0, so the number is positive.
- The exponent bits are 1019 in decimal, so the scale is $2^{1019 - 1023} = 2^{-4} = 1/16$.
- The mantissa is the repeating binary pattern `1001 1001 …`, cut off after 52 bits and rounded up in the last bit.

That rounding is the tiny error behind `0.1 + 0.2 != 0.3`.

In [ ]:
int(bits[1:12], 2)

1019

## Key ideas

- Memory stores bits, grouped into bytes; decimal numbers are stored as **floating point**, with limited precision.
- float64, float32 and float16 keep about 15, 6 and 3 decimal digits. Fewer bits means less precision but less memory.
- A Python float is a 24-byte **boxed** object; a list stores pointers to such objects. A NumPy array stores raw values side by side.
- The CPU keeps recently used data in small, fast **caches**. Memory moves in 64-byte **cache lines**, so reading data in order is much faster than jumping around.